In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import h5py 
import glob
import os
# from downstream import *
import matplotlib.pyplot as plt
import scanpy.external as sce

In [ ]:
base_path = "/home/EOCRC_atlas/"
date = "2026_03_06"
colors = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

# all cell types together 

In [ ]:
adata = sc.read_h5ad(os.path.join(base_path, "data/all_samples_processed_withTier2annotation.h5ad"))

In [ ]:
# Load in CMS predictions
cms = pd.read_csv(os.path.join(base_path, 'results/2026-03-06_CMS_Classification/cms_predictions_CMScaller_03-06-26.csv'), index_col=0)
cms['FRID']=cms.index
cms.head()

In [ ]:
# load metadata 
df = pd.read_csv(os.path.join(base_path, 'results/simple_metadata.csv'))

In [ ]:
# View CMS + metadata dataframe 
cms_df = pd.merge(cms, df, on='FRID', how='inner')
cms_df

In [ ]:
# Print the number of tumors in each cohort with a CMS prediction 
print(len(cms_predicted[cms_predicted['Cohort'] == 'UnderFifty']))
print(len(cms_predicted[cms_predicted['Cohort'] == 'FiftyPlus']))

In [ ]:
# Add CMS Caller classifications to adata 
adata.obs['CMScaller'] = adata.obs['FRID'].astype(str)
adata.obs['CMScaller'] = adata.obs['CMScaller'].astype('category')

adata.obs['CMScaller'] = adata.obs['CMScaller'].cat.add_categories(["CMS1"])
adata.obs['CMScaller'] = adata.obs['CMScaller'].cat.add_categories(["CMS2"])
adata.obs['CMScaller'] = adata.obs['CMScaller'].cat.add_categories(["CMS3"])
adata.obs['CMScaller'] = adata.obs['CMScaller'].cat.add_categories(["CMS4"])

for i in range(cms_df.shape[0]):
    adata.obs.loc[adata.obs['FRID'] == cms_df['FRID'].iloc[i], 'CMScaller'] = cms_df['prediction'].iloc[i]

adata.obs['CMScaller'] = adata.obs['CMScaller'].cat.remove_unused_categories()

In [ ]:
# by patient counts - CMScaller 
sc.set_figure_params(figsize=(2.5, 3))
tmp = pd.crosstab(cms_df['MSI_v2'], cms_df['prediction'], normalize='index') #normalize over each leiden clus
ax = tmp.plot.bar(stacked=True, color=colors).legend(loc='center left',bbox_to_anchor=(1.0, 0.5))#.legend(loc='outside right upper')#.bbox_to_anchor=(.5, 1.0)

fig = ax.get_figure()
plt.show()
fig.savefig(os.path.join(base_path, f'results/{date}_CMS_Classification/CMScaller_barplot_by_MSI_patientCounts_{date}.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# by patient counts - CMScaller 
sc.set_figure_params(figsize=(2.5, 3))
tmp = pd.crosstab(cms_df['Sidedness'], cms_df['prediction'], normalize='index') #normalize over each leiden clus
ax = tmp.plot.bar(stacked=True, color=colors).legend(loc='center left',bbox_to_anchor=(1.0, 0.5))#.legend(loc='outside right upper')#.bbox_to_anchor=(.5, 1.0)

fig = ax.get_figure()
plt.show()
fig.savefig(os.path.join(base_path, f'results/{date}_CMS_Classification/CMScaller_barplot_by_sidedness_patientCounts_{date}.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# by patient counts - CMScaller 
sc.set_figure_params(figsize=(2.5, 3))
tmp = pd.crosstab(cms_df['Cohort'], cms_df['prediction'], normalize='index') #normalize over each leiden clus
ax = tmp.plot.bar(stacked=True, color=colors).legend(loc='center left',bbox_to_anchor=(1.0, 0.5))#.legend(loc='outside right upper')#.bbox_to_anchor=(.5, 1.0)

fig = ax.get_figure()
plt.show()
fig.savefig(os.path.join(base_path, f'results/{date}_CMS_Classification/CMScaller_barplot_by_Cohort_patientCounts_{date}.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# celltype counts - CMScaller 
sc.set_figure_params(figsize=(2.5, 3))
tmp = pd.crosstab(adata.obs['CMScaller'], adata.obs['Annotation_Tier1'], normalize='index') #normalize over each leiden clus
ax = tmp.plot.bar(stacked=True, color=colors).legend(loc='center left',bbox_to_anchor=(1.0, 0.5))#.legend(loc='outside right upper')#.bbox_to_anchor=(.5, 1.0)

fig = ax.get_figure()
plt.show()
#fig.savefig(os.path.join(base_path, f'results/2026-03-06_CMS_Classification/CMS_CMScaller_barplot_by_CellType_{date}.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# function to run chi square test 
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.multitest import multipletests

def chi_square_test(df, var1, var2):
    df = df[pd.notna(df[var1]) & pd.notna(df[var2])]
    grouped = df.groupby([var1, var2]).size().reset_index(name='count')
    contingency_table = grouped.pivot_table(index=var2, columns=var1, values='count', aggfunc='sum', fill_value=0)
    print(f"Contingency Table (Grouped by {var1} and {var2}):")
    print(contingency_table)
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"\nChi-square test p-value: {p}")
    print(f"\nChi-square test dof: {dof}")
    print(f"\nChi-square test chi2: {chi2}")

    print("\nOdds Ratios (UnderFifty vs FiftyPlus):")
    total_fiftyplus = contingency_table['FiftyPlus'].sum()
    total_underfifty = contingency_table['UnderFifty'].sum()

    or_results = []
    for level, row in contingency_table.iterrows():
        a = row['UnderFifty']
        b = row['FiftyPlus']
        c = total_underfifty - a
        d = total_fiftyplus - b

        table_2x2 = [[a, b], [c, d]]
        
        # Fisher's test 
        or_val, p_val = fisher_exact(table_2x2)
        
        # Confidence intervals 
        sm_table = Table2x2(table_2x2)
        ci_low, ci_high = sm_table.oddsratio_confint()
        
        print(f"{level}: OR = {or_val:.3f} (95% CI {ci_low:.3f}–{ci_high:.3f}), p = {p_val:.4g}")
        or_results.append({
            'Level': level,
            'OddsRatio': or_val,
            'CI_lower': ci_low,
            'CI_upper': ci_high,
            'PValue': p_val})
        or_df = pd.DataFrame(or_results)

    # FDR 
    or_df['FDR'] = multipletests(or_df['PValue'], method='fdr_bh')[1]

    print("")
    return contingency_table, p, chi2, dof, or_df

In [ ]:
# CMScaller 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(cms_df, 'Cohort', 'prediction')
or_df

In [ ]:
# look at CMS caller subtypes associated with age using regression - CMS2
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import scale

# create scaled age column 
cms_df['age_scaled'] = scale(cms_df['Age'])

# remove unknown columns 
cms_df_noUNK = cms_df[cms_df['Overall_Stage'] != 'Not Applicable']
cms_df_noUNK = cms_df_noUNK[cms_df_noUNK['MSI_v2'] != 'Unknown']

# make CMS2 binary 
cms_df_noUNK['is_CMS2'] = (cms_df_noUNK['prediction'] == 'CMS2').astype(int)

# fit regresion 
formula = 'is_CMS2 ~ age_scaled + C(MSI_v2) + C(Overall_Stage) + C(Sidedness) + C(Sex) + C(Therapy_v2)'
model_binary = smf.logit(formula, data=cms_df_noUNK).fit()
print(model_binary.summary())

In [ ]:
# look at CMS caller subtypes associated with age using regression - CMS1
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import scale

# create scaled age column 
cms_df['age_scaled'] = scale(cms_df['Age'])

# remove unknown columns 
cms_df_noUNK = cms_df[cms_df['Overall_Stage'] != 'Not Applicable']
cms_df_noUNK = cms_df_noUNK[cms_df_noUNK['MSI_v2'] != 'Unknown']

# make CMS2 binary 
cms_df_noUNK['is_CMS1'] = (cms_df_noUNK['prediction'] == 'CMS1').astype(int)

# fit regresion 
formula = 'is_CMS1 ~ age_scaled + C(MSI_v2) + C(Overall_Stage) + C(Sidedness) + C(Sex) + C(Therapy_v2)'
model_binary = smf.logit(formula, data=cms_df_noUNK).fit()
print(model_binary.summary())

In [ ]:
# look at CMS caller associations subsetted to MSS Left
cms_df_subset = cms_df[(cms_df['MSI_v2'] == "MSS: STABLE") & (cms_df['Sidedness']=='Left')]
contingency_table, p_value, chi2, dof, or_df = chi_square_test(cms_df_subset, 'Cohort', 'prediction')
or_df

In [ ]:
# look at CMS caller subtypes associated with age using regression - MSS left only 
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import scale

cms_df_subset['age_scaled'] = scale(cms_df_subset['Age'])

# remove unknown columns 
cms_df_noUNK = cms_df_subset[cms_df_subset['Overall_Stage'] != 'Not Applicable']
cms_df_noUNK = cms_df_noUNK[cms_df_noUNK['MSI_v2'] != 'Unknown']

# make CMS2 binary 
cms_df_noUNK['is_CMS2'] = (cms_df_noUNK['prediction'] == 'CMS2').astype(int)

# fit regresion 
formula = 'is_CMS2 ~ age_scaled + C(Overall_Stage) + C(Sidedness) + C(Sex) + C(Therapy_v2)'
model_binary = smf.logit(formula, data=cms_df_noUNK).fit()
print(model_binary.summary())